Prévoir 4-5 heures d'entrainement.

On reprend le code d'Andrew

Imports et préparation des données

In [1]:
import os
import pandas as pd
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split

# 1. Chargement des données
# (Assure-toi que les chemins vers le dossier 'data' sont corrects par rapport à ton notebook)
p_attr = pd.read_json("data/Face4Shifts/Anno/p_attr.json", lines=True)

# 2. Création de la variable proxy
p_attr["label"] = (p_attr["long_hair"] == 1) & ((p_attr["smile_with_closed_lips"] == 1) | (p_attr["smile_with_open_lips"] == 1))

# 3. Séparation des données
# On garde le random_state=42 ICI pour que le jeu de test soit identique à tes essais précédents
X = p_attr.drop(columns=["label"])
y = p_attr["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Taille du jeu d'entraînement : {len(X_train)} images")

Taille du jeu d'entraînement : 24000 images


Définition du Dataset

In [2]:
class FaceDataset(Dataset):
    def __init__(self, img_dir, data, label, transform=None):
        self.img_dir = img_dir
        self.data = data
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.iloc[idx]["ID"]
        label = self.label.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{img_id.strip()}.jpg")
        
        # Ouvre l'image et s'assure qu'elle est en RGB
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# Transformations d'entraînement (identiques à l'original)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Création du DataLoader d'entraînement
train_dataset = FaceDataset(img_dir="data/Face4Shifts/Img/Photo", data=X_train, label=y_train, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

La boucle d'entraînement Deep Ensembles

L'important est de ne surtout pas fixer de seed aléatoire globale (torch.manual_seed(...)).

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Entraînement sur : {device}")

# Paramètres globaux
NUM_MODELS = 3
NUM_EPOCHS = 5

for k in range(1, NUM_MODELS + 1):
    print(f"\n{'='*40}")
    print(f" DÉMARRAGE DE L'ENTRAÎNEMENT DU MODÈLE {k}/{NUM_MODELS}")
    print(f"{'='*40}")
    
    # 1. On instancie un TOUT NOUVEAU modèle à chaque fois pour avoir une initialisation aléatoire
    model = models.efficientnet_b0(weights="EfficientNet_B0_Weights.DEFAULT")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    model = model.to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # 2. Boucle d'entraînement classique
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        
        for i, (images, labels) in enumerate(train_dataloader):
            images, labels = images.to(device), labels.float().to(device)
            
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
            # Affichage optionnel tous les 100 batchs pour suivre l'avancement
            if i % 100 == 0 and i > 0:
                print(f"   Modèle {k} | Epoch {epoch+1}/{NUM_EPOCHS} | Batch {i} | Loss: {loss.item():.4f}")
                
        epoch_loss = running_loss / len(train_dataset)
        print(f" -> Modèle {k} | FIN Epoch {epoch+1} | Loss moyenne: {epoch_loss:.4f}")
        
    # 3. Sauvegarde du modèle spécifique
    model_path = f"./model_ens_{k}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"✅ Modèle {k} sauvegardé sous : {model_path}")

print("\nEntraînement de l'Ensemble terminé !")

Entraînement sur : cpu

 DÉMARRAGE DE L'ENTRAÎNEMENT DU MODÈLE 1/3
   Modèle 1 | Epoch 1/5 | Batch 100 | Loss: 0.2779
   Modèle 1 | Epoch 1/5 | Batch 200 | Loss: 0.3481
   Modèle 1 | Epoch 1/5 | Batch 300 | Loss: 0.2084
   Modèle 1 | Epoch 1/5 | Batch 400 | Loss: 0.1332
   Modèle 1 | Epoch 1/5 | Batch 500 | Loss: 0.2942
   Modèle 1 | Epoch 1/5 | Batch 600 | Loss: 0.2683
   Modèle 1 | Epoch 1/5 | Batch 700 | Loss: 0.2157
 -> Modèle 1 | FIN Epoch 1 | Loss moyenne: 0.2682
   Modèle 1 | Epoch 2/5 | Batch 100 | Loss: 0.1341
   Modèle 1 | Epoch 2/5 | Batch 200 | Loss: 0.0620
   Modèle 1 | Epoch 2/5 | Batch 300 | Loss: 0.1095
   Modèle 1 | Epoch 2/5 | Batch 400 | Loss: 0.0999
   Modèle 1 | Epoch 2/5 | Batch 500 | Loss: 0.2327
   Modèle 1 | Epoch 2/5 | Batch 600 | Loss: 0.2815
   Modèle 1 | Epoch 2/5 | Batch 700 | Loss: 0.1170
 -> Modèle 1 | FIN Epoch 2 | Loss moyenne: 0.1757
   Modèle 1 | Epoch 3/5 | Batch 100 | Loss: 0.0710
   Modèle 1 | Epoch 3/5 | Batch 200 | Loss: 0.1089
   Modèle 1 | Epo